### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 3

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _, CRATE, DRATE = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)
INEFF_BATT = 0.95
INEFF_EXT = np.full(I, 0.99)
# INEFF_EXT = np.random.uniform(0.95, 0.98, I)
print("INEFF_EXT:", INEFF_EXT)

✅ 총 10개 파일을 불러왔습니다: 1033.csv, 1818.csv, 2502.csv, 2503.csv, 2634.csv, 2698.csv, 2816.csv, 545.csv, 665.csv, 690.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=3, M1=3721.01, M2=9383.93
   - 개별 K 값: [200. 200. 400. 600. 100. 600. 300. 200. 300. 200.]
INEFF_EXT: [0.99 0.99 0.99 0.99 0.99 0.99 0.99 0.99 0.99 0.99]


### Individual Optimization (original)

In [2]:
m1 = gp.Model("individual")
m1.setParam("MIPGap", 1e-5)

x_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = m1.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m1.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
m1.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m1.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_ind[i, t] == (1/INEFF_EXT[i]) * yp_ind[i, t, s] - INEFF_EXT[i] * ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= z_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF_BATT <= DRATE[i])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= K[i] - z_ind[i, t, s])
    m1.addConstr(zc_ind[i, t, s]*INEFF_BATT <= CRATE[i])
    m1.addConstr(z_ind[i, t, s] <= K[i])
    m1.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + INEFF_BATT * zc_ind[i, t, s] - zd_ind[i, t, s] / INEFF_BATT)

for i, s in product(range(I), range(S)): m1.addConstr(z_ind[i, 0, s] == K0[i])

m1.optimize()

if m1.status == GRB.OPTIMAL:
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; OBJ_IND = m1.objVal
    # phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-05
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05

Optimize a model with 169000 rows, 121240 columns and 385000 nonzeros
Model fingerprint: 0x9857bb5d
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 97964 rows and 25948 columns
Presolve time: 0.26s
Presolved: 71036 rows, 95292 columns, 327144 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 1.41s

Barrier statistics:
 AA' NZ     : 1.482e+06
 Factor NZ  : 7.981e+06 (roughly 130 MB of memory)
 Factor Ops : 2.

In [3]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=1
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT[i]) * x_sum:>8.2f} {(1/INEFF_EXT[i]) * yp_avg:>8.2f} {INEFF_EXT[i] * ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |    77.72    19.23    22.36     0.00    36.48     0.35    37.52
 8 |   133.63   104.66    15.92     2.58    27.41    11.78    71.81
 9 |   168.47   127.86    28.76     4.53    27.48    11.10    85.45
10 |   177.17   154.05    23.99     9.74    24.87    16.00    99.87
11 |   232.51   194.43    40.69    12.57    26.35    16.39   106.66
12 |   216.98   166.23    52.65     5.51    21.34    17.74   114.43
13 |   499.76     0.00   536.20     0.00     3.57    40.01   116.04
14 |   556.21   256.36   296.06     4.14    22.47    14.54    77.31
15 |   463.06   416.67    98.99    58.67    25.91    19.85    83.35
16 |   169.57   169.04    11.64    12.13    20.22    19.21    87.07
17 |   197.25   188.53    22.21    13.95    19.88    19.42    86.07
18 |   268.54   233.14    53.06    21.69    23.04    19.00    84.51
19 |   105.56    96.76    17.74

### Holistic Optimization (Linear Decision Rule + MILP - M) (original)

In [4]:
m2 = gp.Model("holistic_MILP_M")
m2.setParam("MIPGap", 1e-5)
m2.setParam(GRB.Param.TimeLimit, 1200)

x_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = m2.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z") ; zc_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m2.update()

obj_lin = gp.quicksum(
    P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)
) + gp.quicksum(
    (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
    for i in range(I) for t in range(T) for s in range(S)
)

eps = 1e-8
# quad_reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
obj = obj_lin - eps * quad_reg

# NOTE
m2.setObjective(obj, GRB.MAXIMIZE)
# m2.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m2.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_hol[i, t] == (1 / INEFF_EXT[i]) * yp_hol[i, t, s] - INEFF_EXT[i] * ym_hol[i, t, s] + (1 / INEFF_EXT[i]) * dp_hol[i, t, s] - INEFF_EXT[i] * dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= z_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF_BATT <= DRATE[i])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= K[i] - z_hol[i, t, s])
    m2.addConstr(zc_hol[i, t, s]*INEFF_BATT <= CRATE[i])
    m2.addConstr(z_hol[i, t, s] <= K[i])
    m2.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + INEFF_BATT * zc_hol[i, t, s] - zd_hol[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m2.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = m2.addConstr(
        gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)),
        name=f"balance_{t}_{s}",
    )

m2.optimize()

if m2.status == GRB.OPTIMAL or m2.status == GRB.TIME_LIMIT:
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = m2.objVal
    OBJ_HOL = sum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + sum(
        (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    QUAD_HOL = eps * sum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

    lambda_dual = np.zeros((T, S))
    for t, s in product(range(T), range(S)): 
        lambda_dual[t, s] = balance_constraints[t, s].Pi
        # lower_bound = -P_PN[t, s] / S
        # if lambda_dual[t, s] < lower_bound: lambda_dual[t, s] = lower_bound
    print("Direct dual extraction successful!")

else:
    print(f"⚠️ Model finished with status: {m2.status}")

Set parameter MIPGap to value 1e-05
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1200
MIPGap  1e-05

Optimize a model with 171400 rows, 169240 columns and 481000 nonzeros
Model fingerprint: 0xcf8d1c3e
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 2e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22010 columns
Presolve time: 0.41s
Presolved: 75400 rows, 147230 columns, 431000 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 3.42s
Ordering time: 3.46s

Barrier statistics:
 Dense cols : 230
 AA' NZ     : 4.460e+05
 Factor NZ  : 2.016e+06 (roughly 100 MB of memory)
 Factor Ops : 1.755e+08

In [5]:
for i, t, s in product(range(I), range(T), range(S)):
    # if dm_hol[i, t, s] > 0.0001 and zc_hol[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, zc={zc_hol[i, t, s]}")
    # if ym_hol[i, t, s] > 0.0001 and zc_hol[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, ym={ym_hol[i, t, s]}, zc={zc_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.0001 and dm_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, dm={dm_hol[i, t, s]}")
    if zc_hol[i, t, s] > 0.0001 and zd_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, zc={zc_hol[i, t, s]}, zd={zd_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.0001 and ym_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, ym={ym_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_hol[i, t, s] > 0.0001 and yp_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, yp={yp_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

In [6]:
header = (
    f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n"
    + "-" * 105
)
print(f"\n[HOLISTIC] Objective Value = {OBJ_HOL:.2f}, QUAD_HOL = {QUAD_HOL:>2f}")
print(header)

for t in range(0, 24):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_hol[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_hol[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_hol[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_hol[:, t, s]) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_hol[:, t, s]) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.2f}"
    )


[HOLISTIC] Objective Value = 6217394.60, QUAD_HOL = 0.031415
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |     0.00
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |     0.00
 2 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |     0.00
 3 |     0.00     0.00     0.00    73.72     0.00     0.00    73.72     0.00     0.00 |    -0.00
 4 |     0.00     0.00     0.00   172.45     0.00     0.00   172.45     0.00    70.03 |    -0.00
 5 |     0.00     0.00     0.00     7.49     0.00     0.00     7.49     0.00   233.86 |    -0.00
 6 |    56.77     0.00     3.92     0.00     7.36     7.36    52.71     0.00   240.98 |    -0.00
 7 |   595.74     0.00    51.39     0.00   148.65   148.

### Individual Replay

In [7]:
data = []
for t in range(T):
    s_fixed = 75
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT': round(P_RT[t, s_fixed], 3), 
        'Lambda': round(-lambda_dual[t, s_fixed] * S, 3), 
        'P_PN': round(P_PN[t, s_fixed], 3)
    })
pd.DataFrame(data)


# data_for_csv = []
# for t in range(T):
#     for s in range(S):
#         row = {"t": t, "s": s, "P_DA": P_DA[t], "P_RT": P_RT[t, s], "P_PN": P_PN[t, s], "P_IN": -lambda_dual[t, s] * S}
#         data_for_csv.append(row)
# df = pd.DataFrame(data_for_csv)
# output_filename = f"optimization_results_{SEED}.csv"; df.to_csv(output_filename, index=False, encoding="utf-8-sig")
# print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

,Time,P_DA,P_RT,Lambda,P_PN
0,0,105.160,27.547,200.858,210.330
1,1,92.620,24.272,172.597,185.250
2,2,86.020,40.512,156.395,172.050
3,3,82.660,62.665,144.123,165.330
4,4,82.280,49.685,132.153,164.550
5,5,86.010,75.320,119.308,172.020
6,6,93.260,32.236,81.825,186.510
7,7,100.660,81.825,81.825,201.330
8,8,117.660,31.947,80.197,235.320
9,9,127.350,63.690,81.825,254.700


In [8]:
m5 = gp.Model("DER_Individual_Replay")
# m5.setParam("MIPGap", 1e-5)

x = m5.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z = m5.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m5.update()

obj_lin = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(
        lambda_dual[t, s] * (dm[i, t, s] - dp[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    ) 
)

eps = 1e-8
# quad_reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s] + dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

obj = obj_lin - eps * quad_reg

m5.setObjective(obj, GRB.MAXIMIZE)
# m5.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m5.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x[i, t] == (1 / INEFF_EXT[i]) * yp[i, t, s] - INEFF_EXT[i] * ym[i, t, s] + (1 / INEFF_EXT[i]) * dp[i, t, s] - INEFF_EXT[i] * dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= z[i, t, s]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= K[i] - z[i, t, s]) ; m5.addConstr(z[i, t, s] <= K[i])
    m5.addConstr(zd[i, t, s]/INEFF_BATT <= DRATE[i]) ; m5.addConstr(zc[i, t, s]*INEFF_BATT <= CRATE[i])
    m5.addConstr(z[i, t + 1, s] == z[i, t, s] + INEFF_BATT * zc[i, t, s] - zd[i, t, s] / INEFF_BATT)
for i, s in product(range(I), range(S)): m5.addConstr(z[i, 0, s] == K0[i])

m5.optimize()

if m5.status == GRB.OPTIMAL:
    print(f"Optimal solution found! Objective value: {m5.objVal}")
else:
    print("No optimal solution found.")


x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

OBJ_RE = (
    sum(P_DA[t] * x_re[i, t] for i in range(I) for t in range(T))
    + sum(
        (1 / S) * (P_RT[t, s] * yp_re[i, t, s] - P_PN[t, s] * ym_re[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
    + sum(
        lambda_dual[t, s] * (
            sum(dm_re[i, t, s] for i in range(I)) 
            - sum(dp_re[i, t, s] for i in range(I))
        )
        for t in range(T) for s in range(S)
    )
)
QUAD_RE = eps * sum((1 / S) * (dp_re[i, t, s] * dp_re[i, t, s] + dm_re[i, t, s] * dm_re[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 169000 rows, 169240 columns and 433000 nonzeros
Model fingerprint: 0x43b037a4
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 2e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22010 columns
Presolve time: 0.19s
Presolved: 73000 rows, 147230 columns, 383000 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 0.96s

Barrier statistics:
 AA' NZ     : 1.538e+06
 Factor NZ  : 8.396e+06 (roughly 160 MB of memory)
 Factor Ops : 2.653e+09 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Comp

In [9]:
print(round((1/INEFF_EXT[i]) * x_ind[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_re[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_hol[:,:].sum(),2))

33204.84 37294.55 37292.14


In [10]:
for i, t, s in product(range(I), range(T), range(S)):
    # if dm_re[i, t, s] > 0.0001 and zc_re[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, zc={zc_re[i, t, s]}")
    # if ym_re[i, t, s] > 0.0001 and zc_re[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, ym={ym_re[i, t, s]}, zc={zc_re[i, t, s]}")
    if dp_re[i, t, s] > 0.0001 and dm_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, dm={dm_re[i, t, s]}")
    if zc_re[i, t, s] > 0.0001 and zd_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, zc={zc_re[i, t, s]}, zd={zd_re[i, t, s]}")
    if dp_re[i, t, s] > 0.0001 and ym_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, ym={ym_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_re[i, t, s] > 0.0001 and yp_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, yp={yp_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

In [11]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n" + "-" * 105)
print(f"\n[REPLAY] Objective Value = {OBJ_RE:.2f}, QUAD_RE = {QUAD_RE:>2f}") 
print(header)

for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_re[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_re[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_re[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_re[:, t, s]) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_re[:, t, s]) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.4f}"
    )


[REPLAY] Objective Value = 6217394.60, QUAD_RE = 0.031410
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 2 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 3 |     0.00     0.00     0.00    74.65     0.00     0.00    74.65     0.00     0.00 |   0.0000
 4 |     0.00     0.00     0.00   173.85     0.00     0.00   173.85     0.00    70.92 |   0.0000
 5 |     0.00     0.00     0.00     7.27     0.00     0.00     7.27     0.00   236.08 |  -0.0000
 6 |    56.77     0.00     3.89     0.00     7.14     9.27    54.85     0.00   242.99 |  -2.1290
 7 |   595.74     0.00    51.09     0.00   148.97   148.98 

In [12]:
print("=" * 50); print("AGGREGATOR LOSS ANALYSIS"); print("=" * 50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s]) ; total_demand = np.sum(dm_re[:, t, s]); imbalance = total_demand - total_supply
        lambda_price = -lambda_dual[t, s] * S; loss = imbalance * (lambda_price - P_RT[t, s])
        scenario_losses.append(loss)
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

print("=" * 50) ; print("EXTERNAL GRID LOSS ANALYSIS (Physical)"); print("=" * 50)
total_ext_loss_energy = 0
for i, t, s in product(range(I), range(T), range(S)):
    loss_export_rt = yp_re[i, t, s] * (1 / INEFF_EXT[i] - 1); loss_export_int = dp_re[i, t, s] * (1 / INEFF_EXT[i] - 1); loss_export_da = x_re[i, t] * (1 / INEFF_EXT[i] - 1)
    loss_import_rt = ym_re[i, t, s] * (1 - INEFF_EXT[i]); loss_import_int = dm_re[i, t, s] * (1 - INEFF_EXT[i])
    total_ext_loss_energy += (loss_export_rt + loss_export_int + loss_export_da + loss_import_rt + loss_import_int)
print(f"Total Physical Energy Lost in Grid: {total_ext_loss_energy:.2f} kWh")

print(); print("=" * 60); print("SUMMARY"); print("=" * 60)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Internal Aggregator Loss (Financial): {total_loss:.2f}")
print("Realized Profit (Replay + AggLoss)", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print(); print("=" * 60); print("INDIVIDUAL PROFIT ANALYSIS"); print("=" * 60)
profit_ind = np.zeros(I); profit_re = np.zeros(I); profit_hol = np.zeros(I); profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
    lambda_price = -lambda_dual[t, s] * S
    dm_contribution = dm_re[i, t, s] * lambda_price; dp_contribution = dp_re[i, t, s] * lambda_price
    price_weighted_usage[i] += dm_contribution + dp_contribution
total_price_weighted_usage = np.sum(price_weighted_usage)

for i in range(I):
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t]
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    profit_re[i] = 0
    for t in range(T):
        profit_re[i] += P_DA[t] * x_re[i, t]
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
        lambda_price = -lambda_dual[t, :] * S
        profit_re[i] += np.mean([lambda_price[s] * (dp_re[i, t, s] - dm_re[i, t, s]) for s in range(S)])
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])
        lambda_price = -lambda_dual[t, :] * S
        profit_hol[i] += np.mean([lambda_price[s] * (dp_hol[i, t, s] - dm_hol[i, t, s]) for s in range(S)])

loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0: loss_per_player[i] = total_loss * (price_weighted_usage[i] / total_price_weighted_usage)
    else: loss_per_player[i] = total_loss / I
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

print(f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Alloc Loss':<12} {'Gain ($)':<12} {'Gain (%)':<22}"); print("-" * 125)

total_ind = 0; total_re = 0; total_hol = 0; total_re_adj = 0
for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_percentage_str = (f"(+{percentage_change:.1f}%)"
            if percentage_change >= 0
            else f"({percentage_change:.1f}%)"
        )
    else: final_percentage_str = "(N/A)"

    print(f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f} {final_percentage_str:<22}")

    total_ind += profit_ind[i]; total_re += profit_re[i]; total_hol += profit_hol[i]; total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_percentage_str = (
        f"(+{total_percentage_change:.1f}%)"
        if total_percentage_change >= 0
        else f"({total_percentage_change:.1f}%)"
    )
else: total_final_percentage_str = "(N/A)"

print("-" * 125); print(f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f} {total_final_percentage_str:<22}")

AGGREGATOR LOSS ANALYSIS
EXTERNAL GRID LOSS ANALYSIS (Physical)
Total Physical Energy Lost in Grid: 56168.96 kWh

SUMMARY
Individual Participation Profit 5361630.956006297
Expected Replay Profit 6217394.603022519
Internal Aggregator Loss (Financial): -129.92
Realized Profit (Replay + AggLoss) 6217264.682383419
Holistic Profit 6217394.603183249

INDIVIDUAL PROFIT ANALYSIS
Player   Individual   Replay       Re+Loss      Holistic     Alloc Loss   Gain ($)     Gain (%)              
-----------------------------------------------------------------------------------------------------------------------------
0        380058.60    440593.04    440583.10    440593.04    -9.94        60524.50     (+15.9%)              
1        431965.93    502596.56    502586.56    502596.56    -10.00       70620.63     (+16.3%)              
2        645733.75    747603.98    747587.80    747603.98    -16.18       101854.05    (+15.8%)              
3        977926.55    1120832.16   1120810.03   1120832.16  

In [13]:
def decompose_market_terms(P_DA, P_RT, P_PN, x, yp, ym):
    da_profit = float(np.sum(P_DA[None, :] * x))  # sum_i sum_t
    rt_profit = float(
        np.mean(np.sum(P_RT[None, :, :] * yp, axis=(0, 1)))
    )  # mean_s sum_i sum_t
    penalty_cost = float(
        np.mean(np.sum(P_PN[None, :, :] * ym, axis=(0, 1)))
    )  # mean_s sum_i sum_t (cost, +)
    return da_profit, rt_profit, penalty_cost


# 1) Individual
ind_da, ind_rt, ind_pen = decompose_market_terms(
    P_DA, P_RT, P_PN, x_ind, yp_ind, ym_ind
)
ind_adj = ind_pen  # Individual은 imbalance cost 없음(0)

# 2) Decentralized (Proposed)
dec_da, dec_rt, dec_pen = decompose_market_terms(P_DA, P_RT, P_PN, x_re, yp_re, ym_re)
# dec_imb = -float(total_loss)  # total_loss가 음수면 비용(+), 양수면 비용(-)로 반영됨
dec_imb = 0
dec_adj = dec_pen + dec_imb

# 3) Centralized (Holistic)
cen_da, cen_rt, cen_pen = decompose_market_terms(
    P_DA, P_RT, P_PN, x_hol, yp_hol, ym_hol
)
cen_adj = cen_pen  # centralized에 imbalance cost 따로 넣지 않으면 0

print(
    f"Seed={SEED} | "
    f"IND(DA,RT,ADJ)=({ind_da/1000:.3f}, {ind_rt/1000:.3f}, {ind_adj/1000:.3f}) | "
    f"DEC(DA,RT,ADJ)=({dec_da/1000:.3f}, {dec_rt/1000:.3f}, {dec_adj/1000:.3f}) | "
    f"CEN(DA,RT,ADJ)=({cen_da/1000:.3f}, {cen_rt/1000:.3f}, {cen_adj/1000:.3f})"
)

Seed=3 | IND(DA,RT,ADJ)=(4668.565, 1587.971, 894.906) | DEC(DA,RT,ADJ)=(5255.559, 1184.068, 222.596) | CEN(DA,RT,ADJ)=(5255.230, 1184.265, 222.101)


In [14]:
def summarize_imbalance_for_seed(seed, dp_re, dm_re, lambda_dual, P_RT, S, T):
    # -------- Quantity (MWh): scenario-avg of |imbalance(kWh)| summed over t, then /1000 --------
    imbalance_energy_ts_kwh = []
    for t in range(T):
        vals = []
        for s in range(S):
            total_supply_kwh = float(np.sum(dp_re[:, t, s]))
            total_demand_kwh = float(np.sum(dm_re[:, t, s]))
            imbalance_kwh = total_demand_kwh - total_supply_kwh
            vals.append(abs(imbalance_kwh))
        imbalance_energy_ts_kwh.append(float(np.mean(vals)))
    total_imbalance_mwh = float(np.sum(imbalance_energy_ts_kwh)) / 1000.0

    # -------- Value (|$|): convert prices $/MWh -> $/kWh before multiplying by kWh --------
    total_losses = []
    for t in range(T):
        scenario_losses = []
        for s in range(S):
            total_supply_kwh = float(np.sum(dp_re[:, t, s]))
            total_demand_kwh = float(np.sum(dm_re[:, t, s]))
            imbalance_kwh = total_demand_kwh - total_supply_kwh

            lambda_price_mwh = float(-lambda_dual[t, s] * S)  # $/MWh
            p_rt_mwh = float(P_RT[t, s])  # $/MWh

            delta_price_kwh = (lambda_price_mwh - p_rt_mwh) / 1000.0  # $/kWh
            loss_dollar = imbalance_kwh * delta_price_kwh  # $

            scenario_losses.append(loss_dollar)

        total_losses.append(float(np.mean(scenario_losses)))

    total_loss = float(np.sum(total_losses))  # $
    imbalance_value_mag = total_loss  # |$|

    print(
        f"Seed {seed}: Imbalance Quantity = {total_imbalance_mwh:.3f} MWh | Imbalance Value (|$|) = {imbalance_value_mag:.2f}"
    )


# 사용:
summarize_imbalance_for_seed(SEED, dp_re, dm_re, lambda_dual, P_RT, S, T)

Seed 3: Imbalance Quantity = 0.034 MWh | Imbalance Value (|$|) = -0.13
